# Line-segmentation playground: DIVA-HisDB + U-DIADS-TL (merged)

One notebook for both families -- set `FAMILY` below and everything else follows.
**Output format differs between the two** (this is the one thing to keep straight):

| | DIVA-HisDB | U-DIADS-TL |
|---|---|---|
| GT | pixel mask + PAGE-XML region | binary mask only, no region |
| line separation | `disconnect_components` (seam carving), no baseline GT | ARU-Net fusion + `disconnect_components` (`postproc.py`) |
| prediction format | PAGE-XML (TASK-2 schema) | binary mask |
| scoring tool | official DIVA Line Segmentation Evaluator (Java) | Python "Zottin" metric (`71_misc/evaluate_util.py`) |
| tool's own key names | `PixelIU`/`LinesIU`/`LinesRecall`/`LinesPrecision`/`LinesFMeasure` | `Pixel_IU`/`Line_IU`/`DR`/`RA`/`FM` directly |

Both ultimately report the **same 5-tuple** (`Pixel_IU`, `Line_IU`, `DR`, `RA`, `FM`) --
`evaluate_lines.py` remaps DIVA's key names to this common schema, so the two are
directly comparable despite the different tools. This notebook calls that same
module's per-family functions rather than re-implementing the pipelines.


## Setup

In [1]:
%matplotlib inline
import os, sys, random
from pathlib import Path

import cv2
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "00_data").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
SS_DIR = REPO_ROOT / "50_modelling" / "01_simple_segmentation"
sys.path.insert(0, str(REPO_ROOT / "50_modelling"))
sys.path.insert(0, str(SS_DIR))
sys.path.append(str(REPO_ROOT / "71_misc"))  # evaluate_util (official Zottin metric)

from data.diva_dataset import _resolve_image_path, _resolve_layout
from evaluate import sliding_window_inference
from models import build_model
from postproc import remove_small_objects, disconnect_components, build_fused, run_pipeline, PER_MS_PARAMS
from evaluate_util import evaluate_metrics  # official Zottin / FEST 2025 metric
import evaluate_lines as EL  # per-family full-split eval (same functions the CLI script uses)

print(f"Repo root: {REPO_ROOT}")

I0000 00:00:1785008281.723860   55071 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785008281.760105   55071 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Repo root: /home/artur/Thesis/fsl-tl-manuscripts


I0000 00:00:1785008282.500876   55071 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# ---- Configuration --------------------------------------------------------
FAMILY = "diva"   # "diva" | "udiads"
ENCODER = "resnet34"
# diva subsets:   CB55 | CS18 | CS863
# udiads subsets: Latin14396 | Latin2 | Syr341
SUBSET = "CB55"
SPLIT = "test"
# ----------------------------------------------------------------------------

spec = EL.FAMILIES[FAMILY]
RUN = f"unet_{ENCODER}_{FAMILY}_{SUBSET}"
CKPT = REPO_ROOT / spec["ckpt_root"] / RUN / "best.pth"
CONFIG = SS_DIR / "configs" / ("unet_resnet_diva.yaml" if FAMILY == "diva" else "unet_resnet_u_diads.yaml")

cfg = yaml.safe_load(open(CONFIG))
data_root = str(REPO_ROOT / spec["data_root"])
layout = _resolve_layout(data_root, SUBSET, spec["dataset_family"])[SPLIT]
IMG_DIR, GT_DIR = layout["img"], layout["gt"]
GT_XML_DIR = layout.get("xml")  # DIVA only

print(f"Run: {RUN}")
print(f"Checkpoint exists: {CKPT.exists()}  ({CKPT})")
print(f"Images: {IMG_DIR}")

Run: unet_resnet34_diva_CB55
Checkpoint exists: True  (/home/artur/Thesis/fsl-tl-manuscripts/80_models/01_simple_segmentation/diva-hisdb/segmentation/simple_segmentation/unet_resnet34_diva_CB55/best.pth)
Images: /home/artur/Thesis/fsl-tl-manuscripts/00_data/DIVA-HisDB/CB55/img-CB55/img/public-test


## Load model (+ ARU-Net, U-DIADS only)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(cfg).to(DEVICE)
ckpt = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["state_dict"])
model.eval()
print(f"Loaded epoch {ckpt.get('epoch')}  val_iou={ckpt.get('val_iou', float('nan')):.4f}")

# Load ARU-Net for optional fusion (both families can benefit, though DIVA has no GT baseline)
arunet = None
try:
    from models.arunet import ARUNetWrapper
    arunet_pb = REPO_ROOT / "80_models" / "01_simple_segmentation" / "u-diads-tl" / "pretrained" / "arunet" / "model100_ema.pb"
    if arunet_pb.exists():
        arunet = ARUNetWrapper(str(arunet_pb), device="cpu", scale=0.33)
        print("ARU-Net loaded (optional fusion available)")
except Exception as e:
    print(f"ARU-Net not available: {e}")

stems = sorted(p.stem for p in Path(IMG_DIR).iterdir()
               if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".tif", ".tiff"))
print(f"{len(stems)} pages")

## Single page: inference -> line separation -> visualise

In [ ]:
STEM = random.choice(stems)
img = cv2.cvtColor(cv2.imread(_resolve_image_path(IMG_DIR, STEM)), cv2.COLOR_BGR2RGB)

prob = sliding_window_inference(model, img, crop_size=cfg["training"]["crop_size"], device=DEVICE)

if FAMILY == "diva":
    # DIVA: optionally incorporate ARU-Net baseline for improved line detection
    # (same build_fused as U-DIADS, though DIVA has no GT baseline to truly fuse with).
    # Uses separate_lines_projection (word-merge + projection valley cuts) instead of seam-carve.
    from predict_diva_seamcarve import separate_lines_projection, PRESETS
    params = PRESETS["unified"]
    fused = remove_small_objects((prob > 0.5).astype(np.uint8), min_size=params["min_area"])
    
    # Try to fuse with ARU-Net baseline (optional)
    try:
        if arunet is not None:
            fused = build_fused(img, prob, arunet, baseline_erode=8)
    except Exception as e:
        print(f"  (ARU-Net fusion skipped: {e})")
    
    refined = separate_lines_projection(
        fused, merge_width=params["merge_width"], merge_height=params["merge_height"],
        line_sigma=params["line_sigma"], min_line_distance=params["min_line_distance"],
        peak_frac=params["peak_frac"], valley_ratio=params["valley_ratio"], min_area=params["min_area"],
    )
    print(f"{RUN}  page {STEM}")
    print("  (no quick pixel metric here -- the official DIVA score also clips predictions to")
    print("   the main text region before scoring, which this cell doesn't do; see the next")
    print("   cell for the real, region-clipped, Java-evaluator metrics on the full split.)")
else:
    params = PER_MS_PARAMS.get(SUBSET, PER_MS_PARAMS["Latin14396"])
    disc = {k: params[k] for k in ("line_sigma", "min_line_distance", "peak_frac", "valley_ratio")}
    fused = build_fused(img, prob, arunet)
    refined = run_pipeline(fused, disc)
    gt = (cv2.imread(str(Path(GT_DIR) / f"{STEM}.png")).sum(axis=-1) > 0).astype(np.uint8)
    pix, line, DR, RA, FM = evaluate_metrics(gt, EL.to_label_map(refined))
    print(f"{RUN}  page {STEM}")
    print(f"  Pixel_IU={pix:.4f}  Line_IU={line:.4f}  DR={DR:.4f}  RA={RA:.4f}  FM={FM:.4f}")

fig, ax = plt.subplots(1, 3, figsize=(20, 8))
ax[0].imshow(img); ax[0].set_title("image")
ax[1].imshow(prob, cmap="jet"); ax[1].set_title("U-Net prob")
ax[2].imshow(refined, cmap="gray"); ax[2].set_title("refined")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## Full split evaluation (official tool per family)

Calls `evaluate_lines.py`'s own per-family functions -- DIVA exports PAGE-XML and
runs the Java evaluator; U-DIADS reads cached probability maps
(`preprocessing/cache_predictions.py`) and runs the Zottin metric. Same 5-tuple
output either way, appended to the running results CSV.

In [5]:
if FAMILY == "diva":
    metrics = EL.eval_diva_one(ENCODER, SUBSET, SPLIT)
else:
    metrics = EL.eval_udiads_one(ENCODER, SUBSET, SPLIT, arunet)

print(f"{RUN}: {metrics}")

RESULTS_CSV = REPO_ROOT / "99_evaluation" / "01_simple_segmentation" / "playground_results.csv"
if metrics is not None:
    import pandas as pd
    RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    row = {"family": FAMILY, "encoder": ENCODER, "subset": SUBSET, **metrics}
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode="a", header=not RESULTS_CSV.exists(), index=False)
    print(f"appended -> {RESULTS_CSV}")

[preset=unified] method=projection shape=polygon merge=75x13 ls=8.0 mld=30 pf=0.2 vr=0.5 min_area=800 dilate=0
Loaded /home/artur/Thesis/fsl-tl-manuscripts/80_models/01_simple_segmentation/diva-hisdb/segmentation/simple_segmentation/unet_resnet34_diva_CB55/best.pth (epoch 18, val_iou=0.7523210177387025) — CB55 / test


Inference [CB55/test]:   0%|          | 0/10 [00:00<?, ?it/s]

Inference [CB55/test]:  10%|█         | 1/10 [00:09<01:24,  9.44s/it]

Inference [CB55/test]:  20%|██        | 2/10 [00:17<01:11,  8.91s/it]

Inference [CB55/test]:  30%|███       | 3/10 [00:29<01:09,  9.92s/it]

...

DIVA evaluator:  80%|████████  | 8/10 [01:08<00:16,  8.32s/it]

DIVA evaluator:  90%|█████████ | 9/10 [01:20<00:09,  9.22s/it]

DIVA evaluator: 100%|██████████| 10/10 [01:24<00:00,  8.44s/it]



=== DIVA Line Segmentation Evaluation ===
  LinesIU: 0.6794
  LinesFMeasure: 0.7662
  LinesRecall: 0.7319
  LinesPrecision: 0.8081
  PixelIU: 0.7306
  PixelFMeasure: 0.8359
  PixelPrecision: 0.9182
  PixelRecall: 0.7703
  XML : /home/artur/Thesis/fsl-tl-manuscripts/99_evaluation/01_simple_segmentation/diva-hisdb/lines_eval/unet_resnet34_diva_CB55/pred_xml


unet_resnet34_diva_CB55: {'Pixel_IU': 0.73062, 'Line_IU': 0.67935, 'DR': 0.73187, 'RA': 0.80813, 'FM': 0.76619}
appended -> /home/artur/Thesis/fsl-tl-manuscripts/99_evaluation/01_simple_segmentation/playground_results.csv
